In [1]:
# Cell 1 — Imports and paths
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from pathlib import Path
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
RAW_DIR = PROJECT_DIR / "Data" / "Raw"
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
RESULTS_DIR = PROJECT_DIR / "results" / "phase4_finelabels_DE"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase4_finelabels_DE"
for d in [RESULTS_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Setup complete")

Setup complete


In [2]:
# Cell 1b — Verify correct input files before running any DE 
vdj_prefixes = ("IGHV", "IGLV", "IGKV", "TRAV", "TRBV", "TRGV", "TRDV",
                 "IGHD", "IGHJ", "IGLJ", "IGKJ", "TRAC", "TRBC", "TRGC", "TRDC",
                 "IGHA", "IGHE", "IGHG", "IGHM", "IGLC", "IGKC")

for name, path in [
    ("GSE114725", PROCESSED_DIR / "GSE114725_phase1_v2_clean_rawcounts.h5ad"),
    ("GSE176078", PROCESSED_DIR / "GSE176078_phase1_v2_clean_rawcounts.h5ad"),
]:
    check = sc.read_h5ad(path, backed="r")
    mt_genes = [g for g in check.var_names if g.startswith("MT-")]
    vdj_genes = [g for g in check.var_names if g.startswith(vdj_prefixes)]
    print(f"{name}: {check.n_obs} cells x {check.n_vars} genes, "
          f"MT genes: {len(mt_genes)}, VDJ genes: {len(vdj_genes)}")
    if mt_genes or vdj_genes:
        raise ValueError(f"{name} checkpoint still contains MT/VDJ genes")
    del check

print("\nBoth checkpoints verified clean — safe to proceed with DE")

GSE114725: 44662 cells x 14800 genes, MT genes: 0, VDJ genes: 0
GSE176078: 91425 cells x 27343 genes, MT genes: 0, VDJ genes: 0

Both checkpoints verified clean — safe to proceed with DE


In [3]:
# Cell 2 — Load raw counts + cell_type_fine annotation
def load_raw_with_finelabels(raw_path, finelabels_path, dataset_name):
    print(f"Loading {dataset_name}...")
    raw = sc.read_h5ad(raw_path)
    annotated = sc.read_h5ad(finelabels_path, backed="r")

    sample_vals = raw.X[:100].toarray() if hasattr(raw.X, "toarray") else raw.X[:100]
    is_integer_like = np.allclose(sample_vals, np.round(sample_vals))
    print(f"  Raw counts appear to be integers: {is_integer_like}")
    if not is_integer_like:
        raise ValueError(f"{dataset_name} raw.X does NOT look like integer counts")

    # Only keep cells that have a fine label (excludes low-confidence NaN cells)
    fine_labeled_barcodes = annotated.obs_names[annotated.obs["cell_type_fine"].notna()]
    raw_qc = raw[raw.obs_names.isin(fine_labeled_barcodes)].copy()
    print(f"  Raw: {raw.n_obs} cells -> {raw_qc.n_obs} cells (fine-labeled subset)")

    meta_cols = [c for c in annotated.obs.columns]
    raw_qc.obs = raw_qc.obs.join(annotated.obs[meta_cols], rsuffix="_annotated")

    del raw
    gc.collect()
    return raw_qc

adata1_raw = load_raw_with_finelabels(
    PROCESSED_DIR / "GSE114725_phase1_v2_clean_rawcounts.h5ad",
    PROCESSED_DIR / "GSE114725_phase2_v2_annotated_finelabels.h5ad",
    "GSE114725"
)
adata2_raw = load_raw_with_finelabels(
    PROCESSED_DIR / "GSE176078_phase1_v2_clean_rawcounts.h5ad",
    PROCESSED_DIR / "GSE176078_phase2_v2_annotated_corrected_finelabels.h5ad",
    "GSE176078"
)

print(f"\nGSE114725: {adata1_raw.n_obs} cells x {adata1_raw.n_vars} genes")
print(f"GSE176078: {adata2_raw.n_obs} cells x {adata2_raw.n_vars} genes")

Loading GSE114725...
  Raw counts appear to be integers: True
  Raw: 44662 cells -> 43516 cells (fine-labeled subset)
Loading GSE176078...
  Raw counts appear to be integers: True
  Raw: 91425 cells -> 91425 cells (fine-labeled subset)

GSE114725: 43516 cells x 14800 genes
GSE176078: 91425 cells x 27343 genes


In [4]:
# ----------------------------
# Cell 3 — Pseudobulk aggregation function (same as corrected notebook 05)
# ----------------------------
def build_pseudobulk(adata_raw, sample_col, cell_type, cell_type_col="cell_type_fine", min_cells=10):
    subset = adata_raw[adata_raw.obs[cell_type_col] == cell_type]
    pseudobulk_samples = []
    sample_ids = []
    for sample_id in subset.obs[sample_col].unique():
        sample_mask = (subset.obs[sample_col] == sample_id).values
        n_cells = sample_mask.sum()
        if n_cells < min_cells:
            continue
        X_sample = subset.X[sample_mask]
        summed = np.asarray(X_sample.sum(axis=0)).flatten()
        pseudobulk_samples.append(summed)
        sample_ids.append(sample_id)
    counts_df = pd.DataFrame(pseudobulk_samples, index=sample_ids, columns=subset.var_names).T
    meta_df = pd.DataFrame({sample_col: sample_ids}, index=sample_ids)
    return counts_df, meta_df

print("Pseudobulk aggregation function ready")

Pseudobulk aggregation function ready


In [5]:
# Cell 4 — PyDESeq2 runner (identical to corrected notebook 05)
def run_pydeseq2(counts_df, meta_df, group_col, group_a, group_b,
                  cell_type, comparison_name, dataset_name, min_genes_expressed=10):
    meta_sub = meta_df[meta_df[group_col].isin([group_a, group_b])].copy()
    meta_sub[group_col] = meta_sub[group_col].astype(str)
    counts_sub = counts_df[meta_sub.index]
    gene_filter = (counts_sub > 0).sum(axis=1) >= min_genes_expressed
    counts_sub = counts_sub[gene_filter]

    n_a = (meta_sub[group_col] == group_a).sum()
    n_b = (meta_sub[group_col] == group_b).sum()
    print(f"  Attempting {dataset_name} | {cell_type} | {comparison_name}: "
          f"{n_a} {group_a} / {n_b} {group_b} samples, {len(counts_sub)} genes after filtering")

    if n_a < 2 or n_b < 2:
        print(f"    SKIPPED — need >=2 samples per group (got {n_a}/{n_b})")
        return None

    counts_for_deseq = counts_sub.T.astype(int)
    try:
        dds = DeseqDataSet(counts=counts_for_deseq, metadata=meta_sub,
                            design_factors=group_col, refit_cooks=True, quiet=True)
        dds.deseq2()
        ds = DeseqStats(dds, contrast=[group_col, group_a, group_b], quiet=True)
        ds.summary()
        results = ds.results_df.copy().sort_values("padj")
        n_sig = (results["padj"] < 0.05).sum()
        print(f"    SUCCESS — {len(results)} genes tested, {n_sig} significant (padj<0.05)")
        return results
    except Exception as e:
        print(f"    FAILED — {type(e).__name__}: {e}")
        return None

print("PyDESeq2 runner ready")

PyDESeq2 runner ready


In [7]:
# ----------------------------
# Cell 5 (fixed) — build a combined patient+tissue sample identifier
# FIRST, so pseudobulk aggregation correctly treats each patient's
# different tissues as separate samples, not collapsed together.
# ----------------------------
adata1_raw.obs["cell_type_fine"] = adata1_raw.obs["cell_type_fine"].astype(str)
adata1_raw.obs["patient_tissue"] = (
    adata1_raw.obs["patient"].astype(str) + "_" + adata1_raw.obs["tissue"].astype(str)
)

viable_114725 = ['Activated CD8 T cells', 'B cells', 'CD4 Activated T cells',
                  'Monocyte-like macrophages', 'NK-like CD8 T cells',
                  'Resting/Resident macrophages', 'True NK cells']

all_results_1_fine = {}

for ct in viable_114725:
    counts_df, meta_df = build_pseudobulk(adata1_raw, "patient_tissue", ct)
    # Extract tissue from the combined identifier for the comparison grouping
    meta_df["tissue"] = meta_df["patient_tissue"].str.rsplit("_", n=1).str[-1]

    results = run_pydeseq2(
        counts_df, meta_df, group_col="tissue", group_a="TUMOR", group_b="NORMAL",
        cell_type=ct, comparison_name="tumor_vs_normal", dataset_name="GSE114725_fine"
    )
    if results is not None:
        all_results_1_fine[ct] = results
        safe_ct = ct.replace("/", "_").replace(" ", "_")
        results.to_csv(RESULTS_DIR / f"GSE114725_finelabels_DE_{safe_ct}_tumor_vs_normal.csv")
        results[results["padj"] < 0.05].to_csv(
            RESULTS_DIR / f"GSE114725_finelabels_DE_{safe_ct}_tumor_vs_normal_significant.csv")

print("\nGSE114725 fine-label DE complete")

  Attempting GSE114725_fine | Activated CD8 T cells | tumor_vs_normal: 8 TUMOR / 4 NORMAL samples, 5548 genes after filtering


C:\Users\annam\AppData\Local\Temp\ipykernel_7320\785348795.py:23: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts=counts_for_deseq, metadata=meta_sub,
Fitting dispersions...
... done in 0.03 seconds.

Fitting MAP dispersions...
... done in 0.05 seconds.

Fitting LFCs...
... done in 0.05 seconds.



    SUCCESS — 5548 genes tested, 1 significant (padj<0.05)
  Attempting GSE114725_fine | B cells | tumor_vs_normal: 8 TUMOR / 3 NORMAL samples, 1602 genes after filtering


C:\Users\annam\AppData\Local\Temp\ipykernel_7320\785348795.py:23: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts=counts_for_deseq, metadata=meta_sub,
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pydeseq2\dds.py:807: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
Fitting dispersions...
... done in 0.01 seconds.

Fitting MAP dispersions...
... done in 0.03 seconds.

Fitting LFCs...
... done in 0.02 seconds.



    SUCCESS — 1602 genes tested, 1 significant (padj<0.05)
  Attempting GSE114725_fine | CD4 Activated T cells | tumor_vs_normal: 8 TUMOR / 3 NORMAL samples, 5781 genes after filtering


...

    SUCCESS — 1481 genes tested, 0 significant (padj<0.05)
  Attempting GSE114725_fine | Resting/Resident macrophages | tumor_vs_normal: 7 TUMOR / 3 NORMAL samples, 876 genes after filtering


C:\Users\annam\AppData\Local\Temp\ipykernel_7320\785348795.py:23: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts=counts_for_deseq, metadata=meta_sub,
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pydeseq2\dds.py:807: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)


    SUCCESS — 876 genes tested, 2 significant (padj<0.05)
  Attempting GSE114725_fine | True NK cells | tumor_vs_normal: 8 TUMOR / 3 NORMAL samples, 2119 genes after filtering


C:\Users\annam\AppData\Local\Temp\ipykernel_7320\785348795.py:23: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts=counts_for_deseq, metadata=meta_sub,
Fitting dispersions...
... done in 0.03 seconds.

Fitting MAP dispersions...
... done in 0.03 seconds.

Fitting LFCs...
... done in 0.04 seconds.



    SUCCESS — 2119 genes tested, 1 significant (padj<0.05)

GSE114725 fine-label DE complete


In [8]:
mac_finelabel_result = all_results_1_fine["Monocyte-like macrophages"]
sig_genes = mac_finelabel_result[mac_finelabel_result["padj"] < 0.05]
print("Monocyte-like macrophages — 37 significant genes:")
print(sig_genes.index.tolist())

# Check specifically for the original headline genes
for gene in ["FN1", "HSPA1A", "HSPA1B"]:
    if gene in mac_finelabel_result.index:
        row = mac_finelabel_result.loc[gene]
        print(f"\n{gene}: log2FC={row['log2FoldChange']:.2f}, padj={row['padj']:.2e}")

Monocyte-like macrophages — 37 significant genes:
['NBPF15', 'PICK1', 'PBX2', 'ITGAM', 'RTF1', 'KLF3', 'ATPAF2', 'ZC3H7A', 'MIS18BP1', 'MIEF1', 'TPR', 'NIPSNAP1', 'LGALS1', 'SMIM14', 'TMF1', 'ALDOA', 'PIAS1', 'PACS1', 'RWDD1', 'OAF', 'SRGAP2', 'MEPCE', 'FOXO3', 'POU2F2', 'LTBR', 'SIRT6', 'CHMP1B', 'RGS1', 'MBP', 'CITED4', 'CRBN', 'AHNAK', 'NCOR1', 'BCLAF1', 'SUGP2', 'MARCH7', 'SUSD3']

HSPA1A: log2FC=4.44, padj=7.02e-02


In [9]:
# Cell 6 — GSE176078: run DE on viable fine categories, per comparison
# Note: GSE176078's sample_col is already "orig.ident" (one sample per
# tumour, no tissue-collapsing risk like GSE114725 had) 
adata2_raw.obs["cell_type_fine"] = adata2_raw.obs["cell_type_fine"].astype(str)

viable_by_comparison = {
    ("TNBC", "ER+"): ['B cells', 'CAFs', 'Cycling epithelial', 'Cytotoxic CD8 T cells (reclassified)',
                       'Effector CD8 T cells', 'Endothelial cells', 'Epithelial (ambiguous)',
                       'GZMK+ CD8 T cells', 'Interferon-Response CD8 T cells', 'Luminal epithelial',
                       'Macrophages', 'Memory T cells', 'NK cells', 'NKT cells', 'PVL', 'Plasma cells',
                       'Resting/Memory-like CD8 T cells', 'Stress-Response/Activated CD8 T cells',
                       'T cells', 'True NK cells'],
    ("HER2+", "ER+"): ['B cells', 'CAFs', 'Cycling epithelial', 'Cytotoxic CD8 T cells (reclassified)',
                         'Effector CD8 T cells', 'Endothelial cells', 'Epithelial (ambiguous)',
                         'GZMK+ CD8 T cells', 'Interferon-Response CD8 T cells', 'Luminal epithelial',
                         'Macrophages', 'Memory T cells', 'NK cells', 'NKT cells', 'PVL',
                         'Resting/Memory-like CD8 T cells', 'Stress-Response/Activated CD8 T cells',
                         'T cells', 'True NK cells'],
    ("TNBC", "HER2+"): ['B cells', 'CAFs', 'Cycling T cells', 'Cycling epithelial',
                          'Cytotoxic CD8 T cells (reclassified)', 'Effector CD8 T cells', 'Endothelial cells',
                          'Epithelial (ambiguous)', 'Exhausted CD8 T cells', 'GZMK+ CD8 T cells',
                          'Interferon-Response CD8 T cells', 'Luminal epithelial', 'Macrophages',
                          'Memory T cells', 'NK cells', 'NKT cells', 'PVL', 'Resting/Memory-like CD8 T cells',
                          'Stress-Response/Activated CD8 T cells', 'T cells', 'True NK cells'],
}

all_results_2_fine = {}

for (group_a, group_b), cell_types in viable_by_comparison.items():
    comparison_name = f"{group_a}_vs_{group_b}"
    for ct in cell_types:
        counts_df, meta_df = build_pseudobulk(adata2_raw, "orig.ident", ct)
        subtype_lookup = adata2_raw.obs.drop_duplicates("orig.ident").set_index("orig.ident")["subtype"]
        meta_df["subtype"] = meta_df["orig.ident"].map(subtype_lookup)

        results = run_pydeseq2(
            counts_df, meta_df, group_col="subtype", group_a=group_a, group_b=group_b,
            cell_type=ct, comparison_name=comparison_name, dataset_name="GSE176078_fine"
        )
        if results is not None:
            all_results_2_fine[(ct, comparison_name)] = results
            safe_ct = ct.replace("/", "_").replace(" ", "_").replace("(", "").replace(")", "")
            results.to_csv(RESULTS_DIR / f"GSE176078_finelabels_DE_{safe_ct}_{comparison_name}.csv")
            results[results["padj"] < 0.05].to_csv(
                RESULTS_DIR / f"GSE176078_finelabels_DE_{safe_ct}_{comparison_name}_significant.csv")

print(f"\nGSE176078 fine-label DE complete — {len(all_results_2_fine)} comparisons run")

  Attempting GSE176078_fine | B cells | TNBC_vs_ER+: 5 TNBC / 8 ER+ samples, 5851 genes after filtering


C:\Users\annam\AppData\Local\Temp\ipykernel_7320\785348795.py:23: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts=counts_for_deseq, metadata=meta_sub,
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pydeseq2\dds.py:807: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
Fitting dispersions...
... done in 0.05 seconds.

Fitting MAP dispersions...
... done in 0.03 seconds.

Fitting LFCs...
... done in 0.04 seconds.



    SUCCESS — 5851 genes tested, 79 significant (padj<0.05)
  Attempting GSE176078_fine | CAFs | TNBC_vs_ER+: 10 TNBC / 10 ER+ samples, 13706 genes after filtering


C:\Users\annam\AppData\Local\Temp\ipykernel_7320\785348795.py:23: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts=counts_for_deseq, metadata=meta_sub,
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pydeseq2\dds.py:807: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
Fitting dispersions...
... done in 0.43 seconds.

Fitting MAP dispersions...
... done in 0.44 seconds.

Fitting LFCs...
... done in 0.44 seconds.



    SUCCESS — 13706 genes tested, 11 significant (padj<0.05)
  Attempting GSE176078_fine | Cycling epithelial | TNBC_vs_ER+: 8 TNBC / 7 ER+ samples, 11321 genes after filtering


...

    SUCCESS — 2340 genes tested, 0 significant (padj<0.05)
  Attempting GSE176078_fine | T cells | TNBC_vs_HER2+: 8 TNBC / 5 HER2+ samples, 9926 genes after filtering


C:\Users\annam\AppData\Local\Temp\ipykernel_7320\785348795.py:23: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts=counts_for_deseq, metadata=meta_sub,
Fitting dispersions...
... done in 0.17 seconds.

Fitting MAP dispersions...
... done in 0.17 seconds.

Fitting LFCs...
... done in 0.16 seconds.



    SUCCESS — 9926 genes tested, 2 significant (padj<0.05)
  Attempting GSE176078_fine | True NK cells | TNBC_vs_HER2+: 6 TNBC / 4 HER2+ samples, 3014 genes after filtering


C:\Users\annam\AppData\Local\Temp\ipykernel_7320\785348795.py:23: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts=counts_for_deseq, metadata=meta_sub,
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pydeseq2\dds.py:807: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)


    SUCCESS — 3014 genes tested, 0 significant (padj<0.05)

GSE176078 fine-label DE complete — 53 comparisons run
